# Day 3 — 백테스팅 엔진

**백테스팅(Backtesting)** 이란?  
과거 데이터에 투자 전략을 적용해보고 "이 전략이 실제로 돈을 벌었을까?"를 검증하는 것입니다.

---

## 우리가 테스트할 전략: 골든크로스

이동평균선 교차를 이용한 추세 추종 전략입니다.

```
매수 조건 (골든크로스): MA5가 MA20을 위로 돌파
         → 단기 추세가 장기 추세보다 강해짐 = 상승 신호

매도 조건 (데드크로스): MA5가 MA20을 아래로 돌파
         → 단기 추세가 장기 추세보다 약해짐 = 하락 신호
```

```
가격
 │      ↗ 매수                  ↘ 매도
 │    ╱  (골든크로스)          ╲  (데드크로스)
 │─MA5──────────────────────────────
 │─MA20─────────────────────────────
 └─────────────────────────────────→ 시간
```

In [ ]:
%run 01_data_pipeline.ipynb
# Day 1의 함수들(get_data, add_indicators 등)과 라이브러리를 가져옴

## 1. BTC 365일 데이터 준비

백테스팅은 더 긴 기간(365일)이 필요합니다.  
Day 1의 `get_data()`는 200일치만 가져오므로, 여기서는 별도로 365일치를 수집합니다.

In [ ]:
# BTC 365일 일봉 데이터 수집
print("KRW-BTC 365일 데이터 수집 중...")
btc_raw = pyupbit.get_ohlcv("KRW-BTC", count=365)

# 지표 계산 (add_indicators: Day 1 함수 재사용)
df_btc = add_indicators(btc_raw)

print(f"수집 완료: {len(df_btc)}일치")
print(f"기간: {df_btc.index[0].strftime('%Y-%m-%d')} ~ {df_btc.index[-1].strftime('%Y-%m-%d')}")
df_btc[["close", "MA5", "MA20"]].tail(3)

## 2. 백테스팅 설정

In [ ]:
# ── 백테스팅 설정 ────────────────────────────────────────────────
BACKTEST_CONFIG = {
    "ticker":   "KRW-BTC",
    "capital":  1_000_000,   # 초기 자본금 100만 원
    "fee_rate": 0.0005,      # 수수료 0.05% (매수/매도 각각)
    "short_ma": 5,           # 단기 이동평균 (MA5)
    "long_ma":  20,          # 장기 이동평균 (MA20)
}

print("백테스팅 설정:")
for k, v in BACKTEST_CONFIG.items():
    print(f"  {k}: {v}")

## 3. 매매 신호 생성

골든크로스 / 데드크로스 감지 방법:

```
오늘: MA5 ≥ MA20  AND  어제: MA5 < MA20  → 골든크로스 (매수)
오늘: MA5 ≤ MA20  AND  어제: MA5 > MA20  → 데드크로스 (매도)
```

`shift(1)`: 한 행 뒤로 밀기 = "어제" 값을 가져오는 방법

In [ ]:
def generate_signals(df, short_col="MA5", long_col="MA20"):
    """
    골든크로스(매수)와 데드크로스(매도) 신호를 생성합니다.

    추가되는 컬럼:
        signal: 1 = 매수 신호, -1 = 매도 신호, 0 = 신호 없음
    """
    df = df.copy()

    # shift(1) : 1행 앞의 값을 가져옴 = 어제 값
    prev_short = df[short_col].shift(1)
    prev_long  = df[long_col].shift(1)

    # 골든크로스: 어제는 MA5 < MA20 이었는데, 오늘 MA5 ≥ MA20
    golden_cross = (prev_short < prev_long) & (df[short_col] >= df[long_col])

    # 데드크로스: 어제는 MA5 > MA20 이었는데, 오늘 MA5 ≤ MA20
    dead_cross = (prev_short > prev_long) & (df[short_col] <= df[long_col])

    df["signal"] = 0                    # 기본값: 신호 없음
    df.loc[golden_cross, "signal"] = 1  # 매수 신호
    df.loc[dead_cross,   "signal"] = -1 # 매도 신호

    buy_cnt  = golden_cross.sum()
    sell_cnt = dead_cross.sum()
    print(f"신호 생성: 매수 {buy_cnt}회, 매도 {sell_cnt}회")
    return df

In [ ]:
def simulate_trades(df, initial_capital, fee_rate):
    """
    신호에 따라 매수/매도를 시뮬레이션합니다.

    규칙:
      - 보유 중 매수 신호 → 무시
      - 미보유 중 매도 신호 → 무시
      - 기간 종료 시 보유 중이면 마지막 종가로 강제 청산

    반환값:
        trades   : 거래 내역 리스트
        final_capital : 최종 잔고
    """
    capital  = initial_capital  # 현재 보유 현금
    qty      = 0.0              # 보유 BTC 수량
    position = 0                # 0: 미보유, 1: 보유 중
    trades   = []               # 거래 기록

    for date, row in df.iterrows():
        price  = row["close"]
        signal = row["signal"]

        # ── 매수 ─────────────────────────────────────────────────
        if signal == 1 and position == 0:
            # 수수료 차감 후 투자 가능 금액
            invested = capital * (1 - fee_rate)
            qty      = invested / price
            trades.append({
                "type": "매수", "date": date,
                "price": price, "qty": qty, "amount": capital,
                "return": None
            })
            capital  = 0      # 전액 투자
            position = 1

        # ── 매도 ─────────────────────────────────────────────────
        elif signal == -1 and position == 1:
            revenue   = qty * price * (1 - fee_rate)  # 수수료 차감
            buy_amt   = trades[-1]["amount"]           # 매수 시 투자금
            trade_ret = (revenue - buy_amt) / buy_amt * 100

            trades.append({
                "type": "매도", "date": date,
                "price": price, "qty": qty, "amount": revenue,
                "return": trade_ret
            })
            capital  = revenue
            qty      = 0
            position = 0

    # ── 기간 종료 강제 청산 ───────────────────────────────────────
    if position == 1:
        last_price = df["close"].iloc[-1]
        revenue    = qty * last_price * (1 - fee_rate)
        buy_amt    = trades[-1]["amount"]
        trade_ret  = (revenue - buy_amt) / buy_amt * 100
        trades.append({
            "type": "청산", "date": df.index[-1],
            "price": last_price, "qty": qty,
            "amount": revenue, "return": trade_ret
        })
        capital = revenue

    return trades, capital

In [ ]:
def calc_bh_return(df, initial_capital, fee_rate):
    """
    Buy & Hold 수익률을 계산합니다.
    첫날 전량 매수하고 마지막 날 전량 매도하는 전략.
    """
    start_price = df["close"].iloc[0]
    end_price   = df["close"].iloc[-1]

    invested = initial_capital * (1 - fee_rate)  # 매수 수수료
    qty      = invested / start_price
    revenue  = qty * end_price * (1 - fee_rate)  # 매도 수수료

    return (revenue - initial_capital) / initial_capital * 100

In [ ]:
def calc_trade_stats(trades, initial_capital, final_capital):
    """
    거래 통계(승률, 평균 수익/손실 등)를 계산합니다.
    """
    # 매도/청산 거래만 뽑기 (수익률 있는 것)
    closed = [t for t in trades if t["return"] is not None]

    if not closed:
        return {}

    wins   = [t for t in closed if t["return"] >= 0]
    losses = [t for t in closed if t["return"] < 0]

    total_ret = (final_capital - initial_capital) / initial_capital * 100

    # 자산 곡선 재현 (MDD 계산용)
    capital_curve = [initial_capital]
    cap = initial_capital
    for t in trades:
        if t["return"] is not None:
            cap = t["amount"]
            capital_curve.append(cap)
    cap_series = pd.Series(capital_curve)
    mdd = calc_mdd(cap_series) if len(capital_curve) > 1 else 0

    return {
        "total_ret":  total_ret,
        "mdd":        mdd,
        "total_trades": len(closed),
        "wins":       len(wins),
        "losses":     len(losses),
        "win_rate":   len(wins) / len(closed) * 100 if closed else 0,
        "avg_win":    np.mean([t["return"] for t in wins]) if wins else 0,
        "avg_loss":   np.mean([t["return"] for t in losses]) if losses else 0,
    }

In [ ]:
def print_trade_history(trades):
    """거래 내역 표를 출력합니다."""
    print("=== 거래 내역 ===")
    header = f"{'#':>3}  {'유형':4}  {'날짜':12}  {'단가':>18}  {'수량':>12}  {'금액':>16}  {'수익률':>8}"
    print(header)
    print("-" * len(header))

    n = 1
    for t in trades:
        date_str = t["date"].strftime("%Y-%m-%d")
        ret_str  = f"({t['return']:+.2f}%)" if t["return"] is not None else ""
        print(f"{n:>3}  {t['type']:4}  {date_str:12}  {t['price']:>18,.0f} 원  "
              f"{t['qty']:>12.5f}  {t['amount']:>14,.0f} 원  {ret_str:>8}")
        n += 1

    buy_cnt  = sum(1 for t in trades if t["type"] == "매수")
    sell_cnt = sum(1 for t in trades if t["type"] in ("매도", "청산"))
    print(f"\n총 {len(trades)}건 거래 (매수 {buy_cnt} / 매도 및 청산 {sell_cnt})")

In [ ]:
def print_backtest_summary(stats, bh_return, initial_capital, final_capital, df):
    """백테스팅 성과 요약을 출력합니다."""
    days = len(df)
    excess = stats["total_ret"] - bh_return

    print("=== 백테스팅 결과 ===")
    print(f"기간             : {days}일")
    print(f"초기 자본        : {initial_capital:,} 원")
    print(f"최종 자산        : {final_capital:,.0f} 원")
    print(f"총 수익률        : {stats['total_ret']:+.2f}%")
    print(f"MDD             : {stats['mdd']:.2f}%")
    print(f"총 거래          : {stats['total_trades']}회 (매수 {stats['wins'] + stats['losses']} / 매도 {stats['total_trades']})")
    print(f"승률            : {stats['win_rate']:.1f}%")
    print(f"평균 수익 거래   : {stats['avg_win']:+.2f}%")
    print(f"평균 손실 거래   : {stats['avg_loss']:+.2f}%")
    print("────────────────────────────")
    print(f"Buy & Hold      : {bh_return:+.2f}%")
    print(f"전략 초과 수익   : {excess:+.2f}%p")

In [ ]:
# ── 실행 ─────────────────────────────────────────────────────────
cfg    = BACKTEST_CONFIG
df_sig = generate_signals(df_btc)

trades, final_capital = simulate_trades(df_sig, cfg["capital"], cfg["fee_rate"])
bh_return = calc_bh_return(df_btc, cfg["capital"], cfg["fee_rate"])
stats     = calc_trade_stats(trades, cfg["capital"], final_capital)

print_trade_history(trades)
print()
print_backtest_summary(stats, bh_return, cfg["capital"], final_capital, df_btc)

## 4. 시각화

- **상단**: BTC 종가 + MA5 + MA20 + 매수/매도 마커
- **하단**: 전략 자산 곡선 vs Buy & Hold 자산 곡선

In [ ]:
def make_trade_marker_traces(df, trades):
    """
    매수/매도 시점을 차트에 마커(삼각형)로 표시합니다.
    매수 → 초록 ▲  /  매도·청산 → 빨강 ▼
    """
    buy_dates  = [t["date"] for t in trades if t["type"] == "매수"]
    buy_prices = [t["price"] for t in trades if t["type"] == "매수"]

    sell_types  = ("매도", "청산")
    sell_dates  = [t["date"] for t in trades if t["type"] in sell_types]
    sell_prices = [t["price"] for t in trades if t["type"] in sell_types]

    buy_trace = go.Scatter(
        x=buy_dates, y=buy_prices,
        mode="markers",
        marker=dict(symbol="triangle-up", color="green", size=12),
        name="매수"
    )
    sell_trace = go.Scatter(
        x=sell_dates, y=sell_prices,
        mode="markers",
        marker=dict(symbol="triangle-down", color="red", size=12),
        name="매도"
    )
    return buy_trace, sell_trace

In [ ]:
def build_equity_curves(df, trades, initial_capital, fee_rate):
    """
    전략 자산 곡선과 Buy & Hold 자산 곡선을 계산합니다.

    반환값:
        strategy_curve : 날짜별 전략 자산 (Series)
        bh_curve       : 날짜별 B&H 자산 (Series)
    """
    # ── Buy & Hold 곡선 ──────────────────────────────────────────
    start_price = df["close"].iloc[0]
    qty_bh      = initial_capital * (1 - fee_rate) / start_price
    bh_curve    = df["close"] * qty_bh  # 매일 qty × 종가

    # ── 전략 곡선 ────────────────────────────────────────────────
    # 거래 이벤트를 날짜별로 매핑
    trade_map = {t["date"]: t for t in trades}
    capital   = initial_capital
    qty       = 0.0
    position  = 0
    values    = []

    for date, row in df.iterrows():
        if date in trade_map:
            t = trade_map[date]
            if t["type"] == "매수":
                qty      = capital * (1 - fee_rate) / row["close"]
                capital  = 0
                position = 1
            elif t["type"] in ("매도", "청산"):
                capital  = qty * row["close"] * (1 - fee_rate)
                qty      = 0
                position = 0

        # 현재 자산 = 현금 + 보유 코인 가치
        current_val = capital + qty * row["close"]
        values.append(current_val)

    strategy_curve = pd.Series(values, index=df.index)
    return strategy_curve, bh_curve

In [ ]:
def make_backtest_chart(df, trades, strategy_curve, bh_curve):
    """
    백테스팅 결과 시각화: 매매 신호 차트 + 자산 곡선 비교
    """
    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=[
            "KRW-BTC 종가 + 이동평균 + 매매 신호",
            "전략 자산 곡선 vs Buy & Hold"
        ],
        vertical_spacing=0.12,
        row_heights=[0.55, 0.45]
    )

    # 상단: 종가선
    fig.add_trace(go.Scatter(
        x=df.index, y=df["close"],
        line=dict(color="black", width=1.2),
        name="종가"
    ), row=1, col=1)

    # 상단: MA5 / MA20
    fig.add_trace(go.Scatter(
        x=df.index, y=df["MA5"],
        line=dict(color="orange", width=1.2), name="MA5"
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=df.index, y=df["MA20"],
        line=dict(color="blue", width=1.2), name="MA20"
    ), row=1, col=1)

    # 상단: 매수/매도 마커
    buy_trace, sell_trace = make_trade_marker_traces(df, trades)
    fig.add_trace(buy_trace, row=1, col=1)
    fig.add_trace(sell_trace, row=1, col=1)

    # 하단: 전략 자산 곡선
    fig.add_trace(go.Scatter(
        x=strategy_curve.index, y=strategy_curve.values,
        line=dict(color="red", width=2), name="전략"
    ), row=2, col=1)

    # 하단: Buy & Hold 자산 곡선
    fig.add_trace(go.Scatter(
        x=bh_curve.index, y=bh_curve.values,
        line=dict(color="gray", width=2, dash="dash"), name="Buy & Hold"
    ), row=2, col=1)

    fig.update_layout(
        title="골든크로스 전략 백테스팅 결과",
        height=850,
        template="plotly_white"
    )
    fig.update_yaxes(tickformat=",", row=2, col=1)
    return fig

In [ ]:
# ── 차트 출력 ────────────────────────────────────────────────────
strategy_curve, bh_curve = build_equity_curves(
    df_btc, trades, cfg["capital"], cfg["fee_rate"]
)
fig = make_backtest_chart(df_sig, trades, strategy_curve, bh_curve)
fig.show()

## 5. 파라미터 최적화 (선택)

MA 조합을 바꿔가며 어떤 조합이 가장 수익률이 높은지 비교합니다.  
최고 수익률 조합을 강조 표시합니다.

In [ ]:
def run_single_backtest(df, short_win, long_win, capital, fee_rate):
    """MA 조합 하나로 백테스팅 실행 후 수익률 반환"""
    # 해당 MA 컬럼 임시 계산
    df_tmp = df.copy()
    df_tmp["MA_S"] = df_tmp["close"].rolling(short_win).mean()
    df_tmp["MA_L"] = df_tmp["close"].rolling(long_win).mean()
    df_tmp = df_tmp.dropna(subset=["MA_S", "MA_L"])

    df_tmp = generate_signals(df_tmp, short_col="MA_S", long_col="MA_L")
    trades_tmp, final_cap = simulate_trades(df_tmp, capital, fee_rate)
    total_ret = (final_cap - capital) / capital * 100
    return total_ret

# 5가지 이상 MA 조합 테스트
MA_COMBINATIONS = [
    (3, 10), (5, 20), (5, 30), (10, 30), (10, 60), (20, 60)
]

print("=== MA 파라미터 최적화 ===")
print(f"{'단기MA':>6}  {'장기MA':>6}  {'수익률':>10}  {'비고':>6}")
print("-" * 38)

results = []
bh = calc_bh_return(df_btc, cfg["capital"], cfg["fee_rate"])

for short_w, long_w in MA_COMBINATIONS:
    ret = run_single_backtest(df_btc, short_w, long_w, cfg["capital"], cfg["fee_rate"])
    results.append({"short": short_w, "long": long_w, "return": ret})

best_ret  = max(r["return"] for r in results)

for r in results:
    mark = " ← 최고" if r["return"] == best_ret else ""
    print(f"  MA{r['short']:>2} / MA{r['long']:>2}    {r['return']:>8.2f}%{mark}")

print(f"\n  Buy & Hold                {bh:>8.2f}%")